# Headline BROJA decomposition, revised protocol

The published headline (k=7, one k-means seed, occupancy-collapsed) does not
identify the atoms. Established 2026-08-22 and recorded in
`results/k_resolution_headline_impact.json`:

* the value channel has **no cluster structure to find** -- its silhouette is
  statistically indistinguishable from a Gaussian null of identical covariance
  (mean gap +0.007; the null scores *higher* at 6 of 14 candidate k), so the
  silhouette-elbow criterion that selected k=7 was fitting noise on half its
  input;
* on a fixed grid the atoms **converge cleanly** as k rises -- redundancy to
  ~24%, unique-to-value to ~35%, synergy to ~35% -- with seed spread shrinking
  monotonically from +-14.7 points at k=4 to +-2.9 at k=20. They are
  identifiable; they were simply not identified at k=7;
* `collapse_for_occupancy` is the dominant source of both instability and bias.
  It is a no-op at k=4-5 and diverges above: by k=20 it reports unique-to-value
  at 86.1% and synergy at 5.1% where the unguarded grid gives 35.4% and 35.6%.

**This notebook runs the corrected protocol**: fixed pre-collapse grid, k=13,
atoms averaged over 5 k-means seeds with the spread reported, BROJA primary and
CCS as the redundancy-definition sensitivity, both databases, all four targets.
Permutation nulls (synergy and the `I_joint` gate) are computed at seed 0.

**Why k=13.** Inside the converged plateau, near-tightest seed spread, and empty
cells still in low single digits -- at k=16 they jump roughly fourfold. Section 1
re-derives the plateau for *each* database rather than transferring MIMIC's,
since the two have different channel dimensionality (26-d vs 12-d) and n.

In [1]:
import sys
sys.path.insert(0, "../src")

import json, time
import numpy as np
import pandas as pd

import mimic_channels as mimic
import eicu_channels as eicu
import quantize as qz
import pid_broja as pb

pd.set_option("display.width", 220)

K = 13
SEEDS = [0, 1, 2, 3, 4]
N_PERM, N_JOBS, BUDGET_S = 1000, 16, 120.0
TARGETS = ["mortality", "gender_female", "age_ge65", "race_binary"]
DBS = [("MIMIC-IV", mimic), ("eICU-CRD", eicu)]

def channels(mod, cc, target):
    if target == "mortality":
        return cc
    demo, _ = mod.demographic_targets()
    return mod.retarget(cc, demo[target], target)

def levels(ch, mod, k, seed):
    """Fixed pre-collapse grid -- collapse_for_occupancy is deliberately NOT applied."""
    v, _ = qz.kmeans_levels(qz.standardize_for_kmeans(ch.Xv), k, seed=seed)
    s, _ = qz.kmeans_levels(qz.standardize_for_kmeans(ch.Xs, sentinel=mod.TIME_SENTINEL), k, seed=seed)
    return v, s

def H(y):
    p = float(np.mean(y)); return -(p*np.log(p) + (1-p)*np.log(1-p))

BASE = {db: mod.complete_case() for db, mod in DBS}
for db, mod in DBS:
    print(f"{db}: {BASE[db]}")

MIMIC-IV: Channels(n=62,455  V=26d  S=26d  arm=complete_case  prevalence=0.1084)
eICU-CRD: Channels(n=127,843  V=12d  S=12d  arm=complete_case  prevalence=0.0941)


## 1. Re-derive the converged plateau, per database

Rather than transfer k=13 from MIMIC-IV, check on each database that the atoms
have flattened by k=13 and that occupancy is still workable. What matters is
that the atoms stop moving and the seed spread stops shrinking -- not that any
particular k is optimal, since for continuous channels there is no interior
optimum to find.

In [2]:
rows = []
for db, mod in DBS:
    cc = BASE[db]
    for k in (8, 10, 13, 16):
        acc = {a: [] for a in ("red", "u_val", "u_str", "syn")}
        empt, conv = [], []
        for sd in SEEDS:
            v, s = levels(cc, mod, k, sd)
            d = pb.decompose(v, s, cc.y, measure="broja")
            for a in acc: acc[a].append(d["atoms_pct_of_joint"][a])
            empt.append(d["n_empty_cells"]); conv.append(d["converged"])
        rows.append({"db": db, "k": k, "cells": k*k*2,
                     "empty_cells": float(np.mean(empt)), "converged": bool(all(conv)),
                     **{f"{a}_mean": float(np.mean(acc[a])) for a in acc},
                     **{f"{a}_sd": float(np.std(acc[a])) for a in acc}})
plateau = pd.DataFrame(rows)
print("mortality target, BROJA, fixed grid, 5 seeds\n")
print(plateau[["db","k","cells","empty_cells","converged",
               "red_mean","red_sd","u_val_mean","u_val_sd","syn_mean","syn_sd"]]
      .to_string(index=False, float_format=lambda x: f"{x:.2f}"))

mortality target, BROJA, fixed grid, 5 seeds

      db  k  cells  empty_cells  converged  red_mean  red_sd  u_val_mean  u_val_sd  syn_mean  syn_sd
MIMIC-IV  8    128         0.80       True     19.90    4.94       44.54      8.51     27.79    3.72
MIMIC-IV 10    200         0.80       True     23.10    0.94       38.98      5.39     30.68    3.40
MIMIC-IV 13    338         2.80       True     23.74    1.60       36.33      4.38     33.86    3.11
MIMIC-IV 16    512        13.00       True     24.60    2.07       38.23      5.36     31.96    3.81
eICU-CRD  8    128         0.20       True     18.83    0.92       58.65      2.33     22.23    1.71
eICU-CRD 10    200         0.60       True     19.04    1.78       57.02      4.55     23.93    2.86
eICU-CRD 13    338         3.40       True     19.63    1.14       55.47      2.72     24.90    1.70
eICU-CRD 16    512         9.20       True     18.66    0.80       55.68      2.12     25.66    1.41


## 2. The decomposition: k=13, fixed grid, 5 seeds, both measures, all targets

In [3]:
def run_cell(db, mod, target, measure):
    cc = BASE[db]
    ch = channels(mod, cc, target)
    keep = None
    if target != "mortality":
        demo, _ = mod.demographic_targets()
        keep = demo[target].reindex(pd.Index(cc.groups)).notna().to_numpy()

    acc = {a: [] for a in ("red", "u_val", "u_str", "syn")}
    ijs, empt, conv = [], [], []
    for sd in SEEDS:
        v, s = levels(cc, mod, K, sd)
        if keep is not None: v, s = v[keep], s[keep]
        d = pb.decompose(v, s, ch.y, measure=measure)
        for a in acc: acc[a].append(d["atoms_pct_of_joint"][a])
        ijs.append(d["mi_plugin_nats"]["I_joint"])
        empt.append(d["n_empty_cells"]); conv.append(d["converged"])

    # nulls at seed 0 only: synergy, plus the I_joint gate via PID additivity
    v, s = levels(cc, mod, K, SEEDS[0])
    if keep is not None: v, s = v[keep], s[keep]
    d0 = pb.decompose(v, s, ch.y, measure=measure)
    null, n_used = pb.permutation_null_budgeted(v, s, ch.y, n_perm=N_PERM, n_jobs=N_JOBS,
                                                budget_s=BUDGET_S, seed=0, measure=measure)
    syn0 = d0["atoms_nats"]["syn"]
    syn_p = float(((null["syn"] >= syn0).sum() + 1) / (n_used + 1))
    ij0 = d0["mi_plugin_nats"]["I_joint"]
    ij_null = sum(null[a] for a in ("red", "u_val", "u_str", "syn"))
    ij_p = float(((ij_null >= ij0).sum() + 1) / (n_used + 1))

    hy = H(ch.y)
    return {"db": db, "target": target, "measure": measure, "k": K, "n": int(len(ch.y)),
            "H_Y": hy, "I_joint_mean": float(np.mean(ijs)), "I_joint_sd": float(np.std(ijs)),
            "pct_of_H_mean": 100*float(np.mean(ijs))/hy,
            **{f"{a}_mean": float(np.mean(acc[a])) for a in acc},
            **{f"{a}_sd": float(np.std(acc[a])) for a in acc},
            "syn_p": syn_p, "I_joint_p": ij_p, "I_joint_gate_pass": bool(ij_p <= 0.05),
            "n_perm_used": n_used, "empty_cells": float(np.mean(empt)),
            "converged_all_seeds": bool(all(conv)), "n_seeds": len(SEEDS)}

out = []
for db, mod in DBS:
    for target in TARGETS:
        for measure in ("broja", "ccs"):
            t0 = time.time()
            r = run_cell(db, mod, target, measure)
            out.append(r)
            print(f"[{db}] {target:14s} {measure:5s} n={r['n']:7,d} "
                  f"I_joint={r['I_joint_mean']:.5f}±{r['I_joint_sd']:.5f} "
                  f"({r['pct_of_H_mean']:5.2f}% of H)  "
                  f"syn={r['syn_mean']:5.1f}±{r['syn_sd']:4.1f}%  p={r['syn_p']:.4f}  "
                  f"gate={'PASS' if r['I_joint_gate_pass'] else 'FAIL'}  ({time.time()-t0:.0f}s)",
                  flush=True)

res = pd.DataFrame(out)

[MIMIC-IV] mortality      broja n= 62,455 I_joint=0.03427±0.00157 ( 9.99% of H)  syn= 33.9± 3.1%  p=0.0010  gate=PASS  (16s)


[MIMIC-IV] mortality      ccs   n= 62,455 I_joint=0.03427±0.00157 ( 9.99% of H)  syn= 29.3± 1.3%  p=0.0010  gate=PASS  (65s)


[MIMIC-IV] gender_female  broja n= 62,455 I_joint=0.03019±0.00347 ( 4.40% of H)  syn= 25.7± 5.2%  p=0.0010  gate=PASS  (16s)


[MIMIC-IV] gender_female  ccs   n= 62,455 I_joint=0.03019±0.00347 ( 4.40% of H)  syn= 20.0± 1.5%  p=0.0010  gate=PASS  (65s)


[MIMIC-IV] age_ge65       broja n= 62,455 I_joint=0.02240±0.00214 ( 3.23% of H)  syn= 26.6± 2.2%  p=0.0010  gate=PASS  (15s)


...

[eICU-CRD] gender_female  ccs   n=127,800 I_joint=0.01898±0.00107 ( 2.75% of H)  syn=  9.9± 0.2%  p=0.0010  gate=PASS  (64s)


[eICU-CRD] age_ge65       broja n=127,843 I_joint=0.02824±0.00136 ( 4.08% of H)  syn= 10.6± 0.8%  p=0.0010  gate=PASS  (15s)


[eICU-CRD] age_ge65       ccs   n=127,843 I_joint=0.02824±0.00136 ( 4.08% of H)  syn= 19.2± 0.9%  p=0.0010  gate=PASS  (64s)


[eICU-CRD] race_binary    broja n=112,401 I_joint=0.00866±0.00037 ( 2.34% of H)  syn= 20.6± 0.7%  p=0.0010  gate=PASS  (15s)


[eICU-CRD] race_binary    ccs   n=112,401 I_joint=0.00866±0.00037 ( 2.34% of H)  syn= 21.9± 0.8%  p=0.0010  gate=PASS  (64s)


## 3. Headline table (BROJA)

In [4]:
b = res[res.measure == "broja"].copy()
def pm(m, s): return [f"{a:.1f} ± {b_:.1f}" for a, b_ in zip(m, s)]
show = pd.DataFrame({
    "db": b.db, "target": b.target, "n": b.n,
    "I_joint x1e-3": [f"{m*1e3:.2f} ± {s*1e3:.2f}" for m, s in zip(b.I_joint_mean, b.I_joint_sd)],
    "% of H(Y)": [f"{v:.2f}" for v in b.pct_of_H_mean],
    "redundant": pm(b.red_mean, b.red_sd), "unq. value": pm(b.u_val_mean, b.u_val_sd),
    "unq. structure": pm(b.u_str_mean, b.u_str_sd), "synergistic": pm(b.syn_mean, b.syn_sd),
    "syn p": [f"{v:.4f}" for v in b.syn_p], "gate": b.I_joint_gate_pass})
print(show.to_string(index=False))

      db        target      n I_joint x1e-3 % of H(Y)  redundant unq. value unq. structure synergistic  syn p  gate
MIMIC-IV     mortality  62455  34.27 ± 1.57      9.99 23.7 ± 1.6 36.3 ± 4.4      6.1 ± 0.9  33.9 ± 3.1 0.0010  True
MIMIC-IV gender_female  62455  30.19 ± 3.47      4.40 10.8 ± 4.7 63.5 ± 9.6      0.0 ± 0.0  25.7 ± 5.2 0.0010  True
MIMIC-IV      age_ge65  62455  22.40 ± 2.14      3.23  8.9 ± 2.8 64.5 ± 3.9     -0.0 ± 0.0  26.6 ± 2.2 0.0010  True
MIMIC-IV   race_binary  46675  10.90 ± 0.99      2.91 23.1 ± 3.8 27.5 ± 7.6     12.1 ± 3.5  37.3 ± 4.9 0.0010  True
eICU-CRD     mortality 127843  22.66 ± 0.84      7.27 19.6 ± 1.1 55.5 ± 2.7     -0.0 ± 0.0  24.9 ± 1.7 0.0010  True
eICU-CRD gender_female 127800  18.98 ± 1.07      2.75  1.9 ± 0.3 88.9 ± 0.7     -0.0 ± 0.0   9.2 ± 0.4 0.0010  True
eICU-CRD      age_ge65 127843  28.24 ± 1.36      4.08  7.0 ± 0.5 82.4 ± 1.1     -0.0 ± 0.0  10.6 ± 0.8 0.0010  True
eICU-CRD   race_binary 112401   8.66 ± 0.37      2.34  9.8 ± 0.8 69.6 ± 

## 4. CCS sensitivity — does the redundancy definition change the story?

In [5]:
c = res[res.measure == "ccs"].copy()
cmp = b[["db","target","red_mean","u_val_mean","u_str_mean","syn_mean"]].merge(
      c[["db","target","red_mean","u_val_mean","u_str_mean","syn_mean"]],
      on=["db","target"], suffixes=("_broja","_ccs"))
for a in ("red","u_val","u_str","syn"):
    cmp[f"{a}_delta"] = cmp[f"{a}_mean_ccs"] - cmp[f"{a}_mean_broja"]
print(cmp[["db","target","syn_mean_broja","syn_mean_ccs","syn_delta",
           "u_str_mean_broja","u_str_mean_ccs","u_str_delta"]]
      .to_string(index=False, float_format=lambda x: f"{x:6.2f}"))
print(f"\nlargest |synergy| disagreement between measures: {cmp.syn_delta.abs().max():.2f} points")

      db        target  syn_mean_broja  syn_mean_ccs  syn_delta  u_str_mean_broja  u_str_mean_ccs  u_str_delta
MIMIC-IV     mortality           33.86         29.34      -4.52              6.07           10.58         4.52
MIMIC-IV gender_female           25.73         20.02      -5.70              0.02            5.72         5.70
MIMIC-IV      age_ge65           26.58         28.17       1.59             -0.00           -1.59        -1.59
MIMIC-IV   race_binary           37.29         37.52       0.24             12.09           11.85        -0.24
eICU-CRD     mortality           24.90         25.58       0.68             -0.00           -0.68        -0.68
eICU-CRD gender_female            9.21          9.89       0.68             -0.00           -0.68        -0.68
eICU-CRD      age_ge65           10.58         19.19       8.62             -0.00           -8.62        -8.62
eICU-CRD   race_binary           20.61         21.86       1.25             -0.00           -1.25        -1.25



## 5. Against the superseded k=7 headline, and against the copula anchor

In [6]:
old = {("MIMIC-IV","mortality"):(11.65,67.41,-0.00,20.94), ("MIMIC-IV","gender_female"):(16.82,51.85,-0.00,31.33),
       ("MIMIC-IV","age_ge65"):(4.45,81.41,-0.00,14.14), ("MIMIC-IV","race_binary"):(27.81,7.40,37.97,26.82),
       ("eICU-CRD","mortality"):(19.96,53.87,1.99,24.18), ("eICU-CRD","gender_female"):(1.50,93.21,-0.00,5.29),
       ("eICU-CRD","age_ge65"):(8.61,79.23,-0.00,12.16), ("eICU-CRD","race_binary"):(8.67,78.35,-0.00,12.98)}
gc = json.load(open("../results/pid_gcmi_paired_channels.json"))["results"]
rows = []
for _, r in b.iterrows():
    o = old[(r.db, r.target)]
    g = gc[f"{r.db}/{r.target}"]
    rows.append({"db": r.db, "target": r.target,
                 "syn_old_k7": o[3], "syn_new_k13": r.syn_mean, "syn_copula": g["atoms_pct_of_joint"]["syn"],
                 "ustr_old_k7": o[2], "ustr_new_k13": r.u_str_mean,
                 "pctH_old": None, "pctH_new": r.pct_of_H_mean, "pctH_copula": g["I_joint_pct_of_H"]})
delta = pd.DataFrame(rows)
print(delta.to_string(index=False, float_format=lambda x: f"{x:7.2f}"))

      db        target  syn_old_k7  syn_new_k13  syn_copula  ustr_old_k7  ustr_new_k13 pctH_old  pctH_new  pctH_copula
MIMIC-IV     mortality       20.94        33.86        5.50        -0.00          6.07     None      9.99        21.45
MIMIC-IV gender_female       31.33        25.73        3.92        -0.00          0.02     None      4.40        13.85
MIMIC-IV      age_ge65       14.14        26.58        3.18        -0.00         -0.00     None      3.23        10.66
MIMIC-IV   race_binary       26.82        37.29        1.26        37.97         12.09     None      2.91        12.68
eICU-CRD     mortality       24.18        24.90        7.76         1.99         -0.00     None      7.27        10.04
eICU-CRD gender_female        5.29         9.21        6.70        -0.00         -0.00     None      2.75         6.70
eICU-CRD      age_ge65       12.16        10.58        1.13        -0.00         -0.00     None      4.08         9.36
eICU-CRD   race_binary       12.98        20.61 

## 6. Save

In [7]:
payload = {
    "protocol": {"k": K, "grid": "fixed pre-collapse (collapse_for_occupancy NOT applied)",
                 "seeds": SEEDS, "n_seeds": len(SEEDS),
                 "primary_measure": "broja", "sensitivity_measure": "ccs",
                 "nulls": f"permutation, seed 0, budget {BUDGET_S}s, up to {N_PERM} draws"},
    "supersedes": ("the k=7 / k=6 single-seed occupancy-collapsed headline in "
                   "results/headline_result*.json -- see results/k_resolution_headline_impact.json"),
    "plateau_check": plateau.to_dict(orient="records"),
    "cells": res.to_dict(orient="records"),
}
with open("../results/headline_broja_k13.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)
print("wrote results/headline_broja_k13.json")
print(f"gate: {int(res.I_joint_gate_pass.sum())}/{len(res)} cells pass; "
      f"BROJA converged on every seed: {bool(res[res.measure=='broja'].converged_all_seeds.all())}")

wrote results/headline_broja_k13.json
gate: 16/16 cells pass; BROJA converged on every seed: True
